# Лабораторная работа: Понижение размерности

## Цель работы

Ознакомиться с основными методами понижения размерности, изучить их применение на практике, сравнить эффективность разных методов и визуализировать результаты.

## Содержание работы

1. PCA на сгенерированных данных
2. PCA на многомерных данных (Breast Cancer)
3. Метод локтя для выбора числа компонент
4. Метод линейного дискриминантного анализа (LDA)
5. Метод LDA для анизотропных классов
6. Метод t-SNE
7. Kernel PCA для нелинейных данных

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, make_blobs, make_circles
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import TSNE
from sklearn.decomposition import KernelPCA

np.random.seed(42)

### 1. PCA на сгенерированных данных

Сгенерируем двумерный датасет с высокой степенью корреляции между признаками и применим PCA.

In [ ]:
n_samples = 100
x = np.random.normal(0, 1, n_samples)  # Первый признак
y = 2 * x + np.random.normal(0, 0.5, n_samples)  # Второй признак (коррелирует с первым)

X = np.column_stack((x, y))

plt.scatter(X[:, 0], X[:, 1], c='blue', edgecolor='k', s=50)
plt.title('Сгенерированный коррелированный датасет')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.show()

In [ ]:
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X)

plt.scatter(X_pca, np.zeros_like(X_pca), c='red', edgecolor='k', s=50)
plt.title('Результат PCA (1 компонента)')
plt.xlabel('Главная компонента')
plt.yticks([])
plt.grid(True)
plt.show()

In [ ]:
print(f"Главная компонента (направление): {pca.components_}")
print(f"Объясненная дисперсия: {pca.explained_variance_ratio_}")

In [ ]:
# Визуализация направления главной компоненты
plt.scatter(X[:, 0], X[:, 1], c='blue', edgecolor='k', s=50)
pc1_direction = pca.components_[0]
plt.quiver(0, 0, pc1_direction[0], pc1_direction[1],
           angles='xy', scale_units='xy', scale=1, color='red', width=0.01)
plt.title('Направление главной компоненты PCA')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.axis('equal')
plt.show()

Посмотрим, как нормировка данных влияет на результат PCA:

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=1)
X_pca_scaled = pca.fit_transform(X_scaled)

plt.scatter(X_pca_scaled, np.zeros_like(X_pca_scaled), c='red', edgecolor='k', s=50)
plt.title('PCA после стандартизации данных')
plt.xlabel('Главная компонента')
plt.yticks([])
plt.grid(True)
plt.show()

print(f"Объясненная дисперсия после масштабирования: {pca.explained_variance_ratio_}")

### 2. PCA на многомерных данных

Применим метод главных компонент к датасету о диагностике рака (30 признаков).

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target
print(f"Форма данных: {X.shape}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)  # Уменьшаем до 2 компонент
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k', s=50)
plt.colorbar(scatter, label='Класс (0=злокачественный, 1=доброкачественный)')
plt.title('PCA датасета Breast Cancer (2 главные компоненты)')
plt.xlabel('Первая главная компонента')
plt.ylabel('Вторая главная компонента')
plt.grid(True)
plt.show()

In [ ]:
print(f"Главные компоненты:
{pca.components_}")

explained_variance = pca.explained_variance_ratio_
print(f"Объясненная дисперсия каждой компоненты: {explained_variance}")
print(f"Суммарная объясненная дисперсия: {sum(explained_variance):.2f}")

### 3. Метод локтя для PCA

Построим зависимость суммарной объясненной дисперсии от количества главных компонент.

In [ ]:
n_components_range = range(1, 31)
explained_variance = []

for n in n_components_range:
    pca = PCA(n_components=n)
    pca.fit(X_scaled)
    explained_variance.append(sum(pca.explained_variance_ratio_))

plt.plot(n_components_range, explained_variance, marker='o')
plt.title('Метод локтя для выбора числа главных компонент')
plt.xlabel('Количество главных компонент')
plt.ylabel('Суммарная объясненная дисперсия')
plt.grid(True)
plt.show()

In [ ]:
# Оптимальное количество компонент (по методу локтя ~6-7)
pca = PCA(n_components=7)
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
print(f"Объясненная дисперсия каждой компоненты: {explained_variance}")
print(f"Суммарная объясненная дисперсия: {sum(explained_variance):.2f}")

# Визуализируем первые две компоненты
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k', s=50)
plt.title('Первые две главные компоненты (PCA, n=7)')
plt.xlabel('Первая главная компонента')
plt.ylabel('Вторая главная компонента')
plt.grid(True)
plt.show()

### 4. Метод линейного дискриминантного анализа (LDA)

LDA учитывает метки классов и выбирает направления проекции, максимизирующие расстояние между классами.

In [ ]:
n_samples = 100

x1 = np.random.normal(2, 1, n_samples)
y1 = np.random.normal(2, 1, n_samples)
x2 = np.random.normal(6, 1, n_samples)
y2 = np.random.normal(6, 1, n_samples)

X_lda_data = np.vstack((np.column_stack((x1, y1)), np.column_stack((x2, y2))))
y_lda_data = np.hstack((np.zeros(n_samples), np.ones(n_samples)))

plt.scatter(X_lda_data[:, 0], X_lda_data[:, 1], c=y_lda_data, cmap='viridis', edgecolor='k', s=50)
plt.title('Исходный датасет для классификации (2 класса)')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.show()

In [ ]:
scaler = StandardScaler()
X_lda_scaled = scaler.fit_transform(X_lda_data)

lda = LDA(n_components=1)
X_lda = lda.fit_transform(X_lda_scaled, y_lda_data)

plt.scatter(X_lda, np.zeros_like(X_lda), c=y_lda_data, cmap='viridis', edgecolor='k', s=50)
plt.title('Проекция LDA (1 компонента)')
plt.xlabel('Линейный дискриминант')
plt.yticks([])
plt.grid(True)
plt.show()

In [ ]:
print(f"Коэффициенты LDA: {lda.coef_}")

# Визуализация направления проекции LDA
plt.scatter(X_lda_scaled[:, 0], X_lda_scaled[:, 1], c=y_lda_data, cmap='viridis', edgecolor='k', s=50)
lda_direction = lda.coef_[0] / np.linalg.norm(lda.coef_[0])
plt.quiver(0, 0, lda_direction[0], lda_direction[1],
           angles='xy', scale_units='xy', scale=1, color='red', width=0.01)
plt.title('Направление проекции LDA')
plt.xlabel('x (scaled)')
plt.ylabel('y (scaled)')
plt.grid(True)
plt.axis('equal')
plt.show()

### 5. Метод LDA для анизотропных классов

Сравним LDA и PCA на данных с анизотропными (растянутыми) классами.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=200, random_state=170, centers=2)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_blobs = np.dot(X_blobs, transformation)

plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', edgecolor='k', s=50)
plt.title('Анизотропные классы')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.show()

In [ ]:
# LDA на анизотропных данных
scaler = StandardScaler()
X_blobs_scaled = scaler.fit_transform(X_blobs)

lda = LDA(n_components=1)
X_blobs_lda = lda.fit_transform(X_blobs_scaled, y_blobs)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1], c=y_blobs, cmap='viridis', edgecolor='k', s=50)
lda_dir = lda.coef_[0] / np.linalg.norm(lda.coef_[0])
plt.quiver(0, 0, lda_dir[0], lda_dir[1],
           angles='xy', scale_units='xy', scale=1, color='red', width=0.01)
plt.title('Направление проекции LDA')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(X_blobs_lda, np.zeros_like(X_blobs_lda), c=y_blobs, cmap='viridis', edgecolor='k', s=50)
plt.title('Проекция LDA')
plt.yticks([])
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# PCA на тех же данных для сравнения
pca = PCA(n_components=1)
X_blobs_pca = pca.fit_transform(X_blobs_scaled)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1], c=y_blobs, cmap='viridis', edgecolor='k', s=50)
pca_dir = pca.components_[0]
plt.quiver(0, 0, pca_dir[0], pca_dir[1],
           angles='xy', scale_units='xy', scale=1, color='red', width=0.01)
plt.title('Направление главной компоненты PCA')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(X_blobs_pca, np.zeros_like(X_blobs_pca), c=y_blobs, cmap='viridis', edgecolor='k', s=50)
plt.title('Проекция PCA')
plt.yticks([])
plt.grid(True)

plt.tight_layout()
plt.show()

### 6. Метод t-SNE

t-SNE часто используется для визуализации кластеров. Рассмотрим его работу на примере датасета для классификации.

In [ ]:
# Используем сгенерированный датасет для классификации
tsne = TSNE(n_components=1, random_state=42)
X_tsne = tsne.fit_transform(X_lda_scaled)

plt.scatter(X_tsne, np.zeros_like(X_tsne), c=y_lda_data, cmap='viridis', edgecolor='k', s=50)
plt.title('t-SNE (1 компонента, perplexity=30)')
plt.xlabel('t-SNE 1')
plt.yticks([])
plt.grid(True)
plt.show()

In [ ]:
# Влияние perplexity
perplexity_values = [5, 30, 50]
plt.figure(figsize=(15, 4))

for i, perplexity in enumerate(perplexity_values):
    tsne = TSNE(n_components=1, perplexity=perplexity, random_state=42)
    X_tsne = tsne.fit_transform(X_lda_scaled)

    plt.subplot(1, 3, i + 1)
    plt.scatter(X_tsne, np.zeros_like(X_tsne), c=y_lda_data, cmap='viridis', edgecolor='k', s=50)
    plt.title(f't-SNE, perplexity={perplexity}')
    plt.xlabel('Главная компонента (t-SNE1)')
    plt.yticks([])
    plt.grid(True)

plt.tight_layout()
plt.show()

### 7. Kernel PCA для нелинейных данных

Kernel PCA применяет нелинейное преобразование с помощью ядерных функций.

In [ ]:
X_circles, y_circles = make_circles(n_samples=500, factor=0.3, noise=0.05, random_state=42)

plt.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', edgecolor='k', s=50)
plt.title('Датасет make_circles (линейно неразделимые классы)')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.axis('equal')
plt.show()

In [ ]:
scaler = StandardScaler()
X_circles_scaled = scaler.fit_transform(X_circles)

kernels = ['linear', 'poly', 'rbf']

plt.figure(figsize=(15, 4))
for i, kernel in enumerate(kernels):
    kpca = KernelPCA(n_components=2, kernel=kernel,
                     gamma=10 if kernel == 'rbf' else None,
                     random_state=42)
    X_kpca = kpca.fit_transform(X_circles_scaled)

    plt.subplot(1, 3, i + 1)
    plt.scatter(X_kpca[:, 0], X_kpca[:, 1], c=y_circles, cmap='viridis', edgecolor='k', s=50)
    plt.title(f'KernelPCA, kernel={kernel}')
    plt.xlabel('KPCA 1')
    plt.ylabel('KPCA 2')
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Влияние гиперпараметра gamma для RBF-ядра
gamma_values = [0.1, 1, 10]

plt.figure(figsize=(15, 4))
for i, gamma in enumerate(gamma_values):
    kpca = KernelPCA(n_components=2, kernel='rbf', gamma=gamma, random_state=42)
    X_kpca = kpca.fit_transform(X_circles_scaled)

    plt.subplot(1, 3, i + 1)
    plt.scatter(X_kpca[:, 0], X_kpca[:, 1], c=y_circles, cmap='viridis', edgecolor='k', s=50)
    plt.title(f'RBF KernelPCA, gamma={gamma}')
    plt.xlabel('KPCA 1')
    plt.ylabel('KPCA 2')
    plt.grid(True)

plt.tight_layout()
plt.show()